In [1]:
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable

# --- CONFIGURATION ---
# Root folder containing multiple projects (each project may contain schema folders).
PROJECTS_ROOT = Path(r"schemas/simal")

# Folder names that commonly contain schema files inside a project.
SCHEMAS_SUBDIR_CANDIDATES = ("simal_schemas", "siml_schemas")

# Input file extensions treated as SiMAL schemas.
SIMAL_EXTENSIONS = {".simal", ".siml", ".txt"}

# Output root folder.
# Example output path: OUT_ROOT/<project>/.../<schema>.siml
OUT_ROOT = Path(r"schemas/dedented_simal")

# Path to the SiMAL tool implementation (simal_cli.py folder)
LANGUAGE_DIR = Path(r"./")

# Preserve heredoc bodies exactly (recommended).
PRESERVE_HEREDOC_BODIES = True

# Max depth for searching schema folders under each project (limits expensive scans).
MAX_SCHEMA_SEARCH_DEPTH = 6

# --- Imports from SiMAL toolchain (version-pinned via LANGUAGE_DIR) ---
if str(LANGUAGE_DIR) not in sys.path:
    sys.path.insert(0, str(LANGUAGE_DIR))

from simal_parser import parse_dsl
from simal_normalize import remove_leading_indentation

In [2]:
@dataclass(frozen=True)
class DedentResult:
    project: str
    total: int
    dedented: int
    failed: int
    failures: list[dict]


def _iter_project_dirs(projects_root: Path) -> Iterable[Path]:
    for child in sorted(projects_root.iterdir()):
        if child.is_dir() and not child.name.startswith('.'):
            yield child


def _depth(relative: Path) -> int:
    # Number of path segments ('.' -> 0)
    return 0 if str(relative) == '.' else len(relative.parts)


def _find_schema_dirs(project_dir: Path) -> list[Path]:
    """Find directories inside a project that likely contain SiMAL schemas."""
    found: list[Path] = []

    # 1) Common direct locations
    for candidate in SCHEMAS_SUBDIR_CANDIDATES:
        direct = project_dir / candidate
        if direct.is_dir():
            found.append(direct)
        schemas_sub = project_dir / 'schemas' / candidate
        if schemas_sub.is_dir():
            found.append(schemas_sub)

    # 2) Shallow search for candidate dir names
    for candidate in SCHEMAS_SUBDIR_CANDIDATES:
        for p in project_dir.rglob(candidate):
            if not p.is_dir():
                continue
            rel = p.relative_to(project_dir)
            if _depth(rel) <= MAX_SCHEMA_SEARCH_DEPTH:
                found.append(p)

    # 3) Fallback: if the project itself contains schema files
    if not found:
        for ext in SIMAL_EXTENSIONS:
            if any(project_dir.glob(f'*{ext}')):
                found.append(project_dir)
                break

    # Deduplicate while preserving order
    dedup: list[Path] = []
    seen: set[str] = set()
    for p in found:
        key = str(p.resolve()).lower()
        if key not in seen:
            seen.add(key)
            dedup.append(p)
    return dedup


def _iter_schema_files(schema_dir: Path) -> Iterable[Path]:
    for p in sorted(schema_dir.rglob('*')):
        if p.is_file() and p.suffix.lower() in SIMAL_EXTENSIONS:
            yield p


def dedent_project(project_dir: Path, projects_root: Path, out_root: Path) -> DedentResult:
    project_name = project_dir.name
    failures: list[dict] = []

    schema_dirs = _find_schema_dirs(project_dir)
    if not schema_dirs:
        return DedentResult(project=project_name, total=0, dedented=0, failed=0, failures=[])

    schema_files: list[Path] = []
    for d in schema_dirs:
        schema_files.extend(list(_iter_schema_files(d)))

    # Deduplicate files (can happen if schema dirs overlap)
    unique: dict[str, Path] = {}
    for p in schema_files:
        unique[str(p.resolve()).lower()] = p
    schema_files = sorted(unique.values(), key=lambda p: str(p))

    dedented = 0
    for in_path in schema_files:
        try:
            content = in_path.read_text(encoding='utf-8')
            normalized = remove_leading_indentation(content, preserve_heredocs=PRESERVE_HEREDOC_BODIES)

            # Verification: parse both original and normalized text
            _ = parse_dsl(content)
            _ = parse_dsl(normalized)

            rel = in_path.relative_to(projects_root)
            out_path = out_root / rel
            out_path.parent.mkdir(parents=True, exist_ok=True)
            out_path.write_text(normalized, encoding='utf-8')
            dedented += 1
        except Exception as e:
            failures.append({'file': str(in_path), 'error': str(e)})

    return DedentResult(
        project=project_name,
        total=len(schema_files),
        dedented=dedented,
        failed=len(failures),
        failures=failures,
    )

In [ ]:
def dedent_all_projects(projects_root: Path, out_root: Path) -> list[DedentResult]:
    out_root.mkdir(parents=True, exist_ok=True)
    results: list[DedentResult] = []
    for project_dir in _iter_project_dirs(projects_root):
        results.append(dedent_project(project_dir, projects_root, out_root))
    return results


def print_report(results: list[DedentResult]) -> None:
    total_projects = len(results)
    total_files = sum(r.total for r in results)
    total_dedented = sum(r.dedented for r in results)
    total_failed = sum(r.failed for r in results)

    print('=' * 80)
    print('DEDENTED SiMAL SCHEMA REPORT')
    print('=' * 80)
    print(f'Projects scanned: {total_projects}')
    print(f'Schemas found:    {total_files}')
    print(f'Dedented:        {total_dedented}')
    print(f'Failed:          {total_failed}')
    print(f'Output root:     {OUT_ROOT}')
    print(f'Preserve heredocs: {PRESERVE_HEREDOC_BODIES}')

    if total_failed:
        print('\nFailures:')
        for r in results:
            for f in r.failures:
                print(f
)


# Run dedent across all projects
results = dedent_all_projects(PROJECTS_ROOT, OUT_ROOT)
print_report(results)

# Optional: list projects with no schemas detected
no_schema = [r.project for r in results if r.total == 0]
if no_schema:
    print('\nProjects with no schemas detected:')
    for name in no_schema:
        print('-', name)